In [12]:
!pip install ib_insync ipykernel pandas requests lxml 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 30.7 MB/s  0:00:00 eta 0:00:01


In [11]:
from ib_insync import *
import pandas as pd
import time

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=1)
print("Connecté ✅")

Connecté ✅


In [13]:
import requests
import io

headers = {'User-Agent': 'Mozilla/5.0'}
r = requests.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', headers=headers)
table = pd.read_html(io.StringIO(r.text))
sp500 = table[0]
tickers = sorted(sp500['Symbol'].tolist())
tickers = [t.replace('.', ' ') for t in tickers]
print(f"{len(tickers)} tickers récupérés")
print(tickers[:15])

503 tickers récupérés
['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES']


In [14]:
resolved = []
failed = []

for i in range(0, len(tickers), 50):
    batch = tickers[i:i+50]
    contracts = [Stock(t, 'SMART', 'USD') for t in batch]
    ib.qualifyContracts(*contracts)
    
    for contract in contracts:
        if contract.conId > 0:
            resolved.append(contract)
        else:
            failed.append(contract.symbol)
    
    print(f"Batch {i//50 + 1}/{len(tickers)//50 + 1}: {len([c for c in contracts if c.conId > 0])}/{ len(batch)} résolus")
    time.sleep(1)

print(f"\n✅ {len(resolved)} contrats résolus")
if failed:
    print(f"❌ {len(failed)} échecs: {failed}")

Batch 1/11: 50/50 résolus
Batch 2/11: 50/50 résolus
Batch 3/11: 50/50 résolus
Batch 4/11: 50/50 résolus
Batch 5/11: 50/50 résolus
Batch 6/11: 50/50 résolus
Batch 7/11: 50/50 résolus
Batch 8/11: 50/50 résolus
Batch 9/11: 50/50 résolus
Batch 10/11: 50/50 résolus
Batch 11/11: 3/3 résolus

✅ 503 contrats résolus
